In [19]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt

file = Path(r"data\raw\gamer1-ppg-2000-01-01.csv")

# Add the date from the filename to each time-of-day value.
match = re.search(r"(\d{4}-\d{2}-\d{2})$", file.stem)
if not match:
    raise ValueError(f"Could not find a date in {file.name}")

df = pd.read_csv(file)
df["timestamp"] = pd.Timestamp(match.group(1)) + pd.to_timedelta(
    df["Time"], errors="coerce"
)
df["Red_Signal"] = pd.to_numeric(df["Red_Signal"], errors="coerce")

invalid_rows = df["timestamp"].isna() | df["Red_Signal"].isna()
print(f"Invalid rows: {invalid_rows.sum():,}")

# Keep the raw files unchanged; aggregate duplicate times only for processing.
df = (
    df.loc[~invalid_rows, ["timestamp", "Red_Signal"]]
      .groupby("timestamp", as_index=False)["Red_Signal"]
      .mean()
      .sort_values("timestamp")
      .reset_index(drop=True)
)

# Find the typical interval between distinct timestamps.
t = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds().to_numpy()
dt = np.diff(t)
median_dt = np.median(dt[dt > 0])
estimated_fs = 1 / median_dt
gap_limit = max(0.1, 5 * median_dt)

print(f"Approximate rate after combining duplicate times: {estimated_fs:.1f} samples/second")
print(f"Gap-splitting threshold: {gap_limit:.3f} seconds")

# Split at gaps rather than interpolating across missing recording time.
breaks = np.flatnonzero(dt > gap_limit)
starts = np.r_[0, breaks + 1]
ends = np.r_[breaks + 1, len(df)]
segments = [(start, end) for start, end in zip(starts, ends) if end - start > 10]

if not segments:
    raise ValueError("No continuous segment long enough to process.")

# Use the longest continuous segment for the example plot.
start, end = max(segments, key=lambda bounds: bounds[1] - bounds[0])
segment = df.iloc[start:end].reset_index(drop=True)
t = (segment["timestamp"] - segment["timestamp"].iloc[0]).dt.total_seconds().to_numpy()
signal = segment["Red_Signal"].to_numpy(dtype=float)

# Filtering works best on evenly spaced samples, so interpolate within this
# continuous segment only. Do not interpolate across the gaps found above.
fs = 1 / np.median(np.diff(t))
t_even = np.arange(t[0], t[-1], 1 / fs)
signal_even = np.interp(t_even, t, signal)

# Initial visualization filter: keep likely pulse-related frequencies.
# This is a starting point to inspect, not a validated heartbeat detector.
if fs <= 10:
    raise ValueError(f"Estimated sample rate ({fs:.1f} Hz) is too low for this filter.")

sos = butter(3, [0.5, 5.0], btype="bandpass", fs=fs, output="sos")
filtered = sosfiltfilt(sos, signal_even)

# Plot the first 10 seconds of the selected continuous segment.
show = t_even <= 10

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(t_even[show], signal_even[show], linewidth=0.8)
axes[0].set_title("Cleaned raw PPG")
axes[0].set_ylabel("Red_Signal")

axes[1].plot(t_even[show], filtered[show], linewidth=0.8)
axes[1].set_title("Filtered PPG (initial 0.5–5 Hz band)")
axes[1].set_xlabel("Seconds from start of selected segment")
axes[1].set_ylabel("Filtered signal")

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'data\\raw\\gamer1-ppg-2000-01-01.csv'